# Portfolio Project: Online Retail Exploratory Data Analysis with Python

## Overview

In this project, you will step into the shoes of an entry-level data analyst at an online retail company, helping interpret real-world data to help make a key business decision.

## Case Study
In this project, you will be working with transactional data from an online retail store. The dataset contains information about customer purchases, including product details, quantities, prices, and timestamps. Your task is to explore and analyze this dataset to gain insights into the store's sales trends, customer behavior, and popular products. 

By conducting exploratory data analysis, you will identify patterns, outliers, and correlations in the data, allowing you to make data-driven decisions and recommendations to optimize the store's operations and improve customer satisfaction. Through visualizations and statistical analysis, you will uncover key trends, such as the busiest sales months, best-selling products, and the store's most valuable customers. Ultimately, this project aims to provide actionable insights that can drive strategic business decisions and enhance the store's overall performance in the competitive online retail market.

## Project Objectives
1. Describe data to answer key questions to uncover insights
2. Gain valuable insights that will help improve online retail performance
3. Provide analytic insights and data-driven recommendations

## Dataset

The dataset you will be working with is the "Online Retail" dataset. It contains transactional data of an online retail store from 2010 to 2011. The dataset is available as a .xlsx file named `Online Retail.xlsx`. This data file is already included in the Coursera Jupyter Notebook environment, however if you are working off-platform it can also be downloaded [here](https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx).

The dataset contains the following columns:

- InvoiceNo: Invoice number of the transaction
- StockCode: Unique code of the product
- Description: Description of the product
- Quantity: Quantity of the product in the transaction
- InvoiceDate: Date and time of the transaction
- UnitPrice: Unit price of the product
- CustomerID: Unique identifier of the customer
- Country: Country where the transaction occurred

## Tasks

You may explore this dataset in any way you would like - however if you'd like some help getting started, here are a few ideas:

1. Load the dataset into a Pandas DataFrame and display the first few rows to get an overview of the data.
2. Perform data cleaning by handling missing values, if any, and removing any redundant or unnecessary columns.
3. Explore the basic statistics of the dataset, including measures of central tendency and dispersion.
4. Perform data visualization to gain insights into the dataset. Generate appropriate plots, such as histograms, scatter plots, or bar plots, to visualize different aspects of the data.
5. Analyze the sales trends over time. Identify the busiest months and days of the week in terms of sales.
6. Explore the top-selling products and countries based on the quantity sold.
7. Identify any outliers or anomalies in the dataset and discuss their potential impact on the analysis.
8. Draw conclusions and summarize your findings from the exploratory data analysis.

## Task 1: Load the Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import data_exploring as expl
import data_imputation as imp
import data_cleaning as cln

In [ ]:
BASE_DIR = Path.cwd().resolve().parent
DATA_DIR = BASE_DIR / "data"
retail_path = DATA_DIR / 'Online Retail.xlsx'

In [ ]:
df = pd.read_excel(retail_path)
display(df.head(5))
display(df.tail(5))

In [ ]:
expl.show_basic_df_info(df)

In [ ]:
print('\nId counts:')
print(f'different customers: {df.CustomerID.nunique()}')
print(f'different products: {df.StockCode.nunique()}')
print(f'duplicated rows {df.duplicated().sum()}')

In [ ]:
expl.get_categorical_counts(df=df, cat_cols=['Country', 'StockCode'])

In [ ]:
# assuming product returns
display(df[df['Quantity'] <= 0])

In [ ]:
display(df[df['Quantity'] == 0])

In [ ]:
# assuming these are broken
display(df[df['UnitPrice'] <= 0])

In [ ]:
display(df[df['UnitPrice'] < 0])

## Task 2: Data cleaning
<b>Step 1:</b> Drop duplicated rows

In [ ]:
df_clean = df.drop_duplicates()

<b>Step 2:</b> Standardize descriptions

In [ ]:
df_clean['CustomerID'] = df_clean['CustomerID'].astype(str)
df_clean = cln.standardize_strs(df=df_clean, str_cols=['Description', 'InvoiceNo', 'StockCode', 'CustomerID'])

### Data imputation
<b>Step 1:</b> Attempting to complete descriptions with product ids

In [ ]:
df_imp = imp.fill_missing_values_by_id(df, id_col='StockCode', value_col='Description')
print(f'{df["Description"].isna().sum()} vs {df_imp["Description"].isna().sum()}')

<b>Step 2:</b> Attempting to complete unit price with product ids

In [ ]:
df_imp['UnitPrice'] = df_imp['UnitPrice'].replace(0.0, np.nan)
df_imp['UnitPrice'].isna().sum()

In [ ]:
df_imp_2 = imp.fill_missing_values_by_id(df_imp, id_col='StockCode', value_col='UnitPrice')
print(f'{df_imp["UnitPrice"].isna().sum()} vs {df_imp_2["UnitPrice"].isna().sum()}')

<b>Step 3:</b> Removing rows without unit price

Too few rows to matter, and UnitPrice is one of the most interesting variables

In [ ]:
df_imp_2 = df_imp_2.dropna(subset=['UnitPrice'])

<b>Step 4:</b> Replacing missing descriptions with missing

Too few rows to matter, and description is one of the least interesting variables

In [ ]:
df_imp_2['Description'] = df_imp_2['Description'].fillna(value='missing')

### Saving clean dataset

In [ ]:
df_imp_2.to_excel('Online Retail Cleaned.xlsx', index=False)

# Conclusions

## Are the results what was expected?
The dataset has less interesting information than anticipated. Some columns have values that are difficult to make sense of without documentation or knowledge of this particular company.


## What are the key findings and takeaways?

- There are duplicate rows

- There are unit prices less or equal to 0. Negative unit prices could be interpreted as accounting adjustments based on descriptions. unit prices with 0 don't make sense, so they were initially imputed based on the product stock and discarded if not possible

- There are quantities less than 0. Negative quantities could be interpreted as items returned or wrongfully sold based on descriptions.

- There are multiple descriptions for a single stock code. This was modified to match the most popular description for each code